# SHAP (TreeExplainer)
**tabular › explain_classification** · notebook `shap_tree`

**Kapan dipakai:** Penjelasan global (fitur mana yang penting dan arah pengaruhnya) dan lokal (mengapa satu baris diprediksi begitu). Bahan utama bagian analisis di laporan.

**Jebakan umum:** Hitungnya berat untuk data besar (pakai subset). Nilai SHAP menjelaskan model, bukan sebab-akibat dunia nyata. Fitur berkorelasi membagi kredit secara tak intuitif.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/classification/titanic_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/classification/titanic_test.csv"      # None kalau tidak ada file test
TARGET     = "survived"
ID_COL     = 'id'
DROP_COLS  = ['alive']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "shap_tree"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "explain_classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model yang akan dijelaskan
Penjelasan selalu dihitung pada **validation**, bukan training.

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
pipe = build_pipeline(model, NUM, CAT, kind='tree').fit(Xtr, ytr)

In [ ]:
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(proba.shape[1]))
display(show_table(metrics))

Matriks fitur setelah preprocessing (nama kolom dipertahankan). `mdl` = model di dalam pipeline.

In [ ]:
from superdb.tabular import feature_names
names = feature_names(pipe)
pre, mdl = pipe.named_steps["pre"], pipe.named_steps["model"]
Ztr = pd.DataFrame(pre.transform(Xtr), columns=names, index=Xtr.index)
Zva = pd.DataFrame(pre.transform(Xva), columns=names, index=Xva.index)
Zva.shape

## 5. SHAP global
Beeswarm: tiap titik = satu baris; sumbu x = dorongan ke prediksi; warna = nilai fitur (merah tinggi). Bar = rata-rata |SHAP|.

In [ ]:
import shap
Zs = Zva.sample(min(500, len(Zva)), random_state=SEED)       # subset agar cepat
sv = shap.TreeExplainer(mdl)(Zs)
if sv.values.ndim == 3:                                      # klasifikasi: ambil kelas terakhir (positif)
    sv = sv[..., -1]
plt.figure(); shap.plots.beeswarm(sv, max_display=15, show=False); plt.tight_layout()
plt.savefig(OUT / "shap_beeswarm.png", dpi=130); plt.show()
plt.figure(); shap.plots.bar(sv, max_display=15, show=False); plt.tight_layout()
plt.savefig(OUT / "shap_bar.png", dpi=130); plt.show()
imp = pd.Series(np.abs(sv.values).mean(0), index=Zs.columns).sort_values(ascending=False)
imp.to_csv(OUT / "shap_importance.csv"); imp.head(10)

## 6. SHAP lokal: mengapa baris ini diprediksi begitu?
Batang merah mendorong prediksi naik, biru menurunkan, mulai dari nilai rata-rata model.

In [ ]:
plt.figure(); shap.plots.waterfall(sv[0], max_display=12, show=False); plt.tight_layout()
plt.savefig(OUT / "shap_waterfall.png", dpi=130, bbox_inches="tight"); plt.show()
top = imp.index[0]
plt.figure(); shap.plots.scatter(sv[:, top], show=False); plt.tight_layout()
plt.savefig(OUT / "shap_dependence.png", dpi=130); plt.show()

In [ ]:
save_json({"valid": metrics, "method": "shap_tree"}, OUT / "metrics.json")
save_model({"pipe": pipe, "cols": cols}, OUT / "model.joblib"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Penjelasan model adalah bahan bab *analisis* di laporan: tulis 3 temuan utama + satu contoh kasus.
- Fitur yang 'terlalu penting' (mis. satu kolom mendominasi) = curigai kebocoran (lihat `notebooks/tabular/validation/`).